# Get the Benquet, Sainsbury et al data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
cd "/app/"

/app


/app/.venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
%run env_aws.py
%run run.py connect

/app/.venv/lib/python3.9/site-packages/datajoint/plugin.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-11 15:52:33,572][INFO]: Connecting admin@vr4mice-ar-collab.cluster-cn54f38qpzgm.eu-central-1.rds.amazonaws.com:3306
[2026-03-11 15:52:33,752][INFO]: Connected admin@vr4mice-ar-collab.cluster-cn54f38qpzgm.eu-central-1.rds.amazonaws.com:3306


In [4]:
from vr4mice.schema.base_analysis import DataFrame
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from vr4mice.analysis import plotting
from vr4mice.schema.interpolated_trajectories import InterpolatedTrials, MeanXYTrajectory, MeanVelocities,YBinnedXYTrajectory
from vr4mice.schema.session_metrics import TrialMetrics
from vr4mice.schema import base_analysis
from vr4mice.analysis.analysis import style
from vr4mice.analysis import analysis
from vr4mice.analysis import utils

from base.base_min_schemas.base_schemas.schemas import exp, mice

from statsmodels.stats.anova import AnovaRM

from scipy.stats import ttest_rel, ttest_ind
import scipy.stats as stats
import warnings
warnings.filterwarnings("ignore")

style()

/app/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
# Root + descendant traversal with conditional session_label/set_name restrictions
import numpy as np
import datajoint as dj
import pandas as pd

from vr4mice.schema.vr4mice import Dataset as VRDataset

# Restriction sets
SESSION_LABELS = [
    "ar_detection_no_velthr", "ar_detection_velthr", "ar_discrim",
    "ar_discrim_5_occluders", "ar_discrim_occluders", "ar_det_no_velthr_inv",
    "ar_detection_velthr_inv", "ar_discrim_inv", "ar_discrim_5_occluders_inv",
    "ar_discrim_occluders_inv",
]

SET_LABELS = ["contrast_white_target", "contrast_black_target"]

SCHEMA_SUFFIXES = [
    "vr4mice",
    "base",
    "base_analysis",
    "decision",
    "dlc",
    #"inputs_videos",
    "interpolated_trajectories",
    "latency_tests",
    "session_metrics",
]

TABLE_DATASET_OVERRIDES = {
    "dlc.__offline_kinematics": "Pheasant_2024-08-21_1",
    "latency_tests.__signals_photodiode_aligned": "Latencytest1_2024-10-31_2",
}

def _load_included_tables(path: str):
    try:
        values = np.load(path, allow_pickle=False)
        values = [str(v) for v in values.tolist()]
        return sorted(set(values))
    except Exception as err:
        print(f"WARNING: could not load include list from {path}: {err}")
        return []

INCLUDED_TABLES_FROM_FILE = _load_included_tables("notebooks/Paper_figures/included_tables.npy")

def _full_table_name(database: str, table_name: str) -> str:
    return f"`{database}`.`{table_name.strip('`')}`"


def _split_full_table_name(full_table_name: str):
    schema_name, table_name = full_table_name.split("`.")
    return schema_name.strip("`"), table_name.strip("`")


def _schema_from_full_table_name(full_table_name: str) -> str:
    # `schema`.`table` -> schema
    return _split_full_table_name(full_table_name)[0]


def _table_suffix_key(full_table_name: str, prefix: str) -> str:
    schema_name, table_name = _split_full_table_name(full_table_name)
    if prefix and schema_name.startswith(prefix):
        schema_name = schema_name[len(prefix):]
    return f"{schema_name}.{table_name}"


def _is_system_table(table_name: str) -> bool:
    # DataJoint internal table uses ~ prefix (e.g. ~log).
    # Keep __* tables because those are valid Computed tables.
    return table_name.startswith("~")


def _sql_quote(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"


def _apply_database_prefix_to_included_tables(prefix: str, included_tables):
    prefixed_tables = []
    for full_table_name in included_tables:
        try:
            schema_name, table_name = _split_full_table_name(full_table_name)
        except ValueError:
            continue

        prefixed_tables.append(_full_table_name(f"{prefix}{schema_name}", table_name))

    return sorted(set(prefixed_tables))


def _infer_schema_databases(base_dataset_database: str, suffixes):
    if base_dataset_database.endswith("vr4mice"):
        prefix = base_dataset_database[: -len("vr4mice")]
    else:
        # Fallback: assume no prefix if naming is unexpected
        prefix = ""

    existing_dbs = {row[0] for row in dj.conn().query("SHOW DATABASES").fetchall()}
    candidate_dbs = [f"{prefix}{suffix}" for suffix in suffixes]
    return [db for db in candidate_dbs if db in existing_dbs], prefix


def _list_schema_tables(schema_databases, included_tables=None):
    included_tables = [] if included_tables is None else included_tables
    included_set = set(included_tables)
    all_tables = []
    for db in schema_databases:
        rows = dj.conn().query(f"SHOW TABLES IN `{db}`").fetchall()
        for row in rows:
            raw_table_name = row[0]
            if _is_system_table(raw_table_name):
                continue
            full_table_name = _full_table_name(db, raw_table_name)
            if included_set and full_table_name not in included_set:
                continue
            all_tables.append(full_table_name)
    return sorted(set(all_tables))


def _fetch_table_storage_stats(schema_databases):
    if not schema_databases:
        return {}

    schema_sql = ", ".join(_sql_quote(db) for db in schema_databases)
    rows = dj.conn().query(
        f"""
        SELECT
            table_schema,
            table_name,
            COALESCE(table_rows, 0) AS table_rows,
            COALESCE(data_length, 0) AS data_length,
            COALESCE(index_length, 0) AS index_length
        FROM information_schema.tables
        WHERE table_schema IN ({schema_sql})
        """
    ).fetchall()

    stats = {}
    for schema_name, table_name, table_rows, data_length, index_length in rows:
        table_rows = int(table_rows or 0)
        data_length = int(data_length or 0)
        index_length = int(index_length or 0)
        table_size_bytes = data_length + index_length
        avg_row_size_bytes = table_size_bytes / table_rows if table_rows > 0 else None
        stats[_full_table_name(schema_name, table_name)] = {
            "table_rows_estimate": table_rows,
            "data_size_bytes": data_length,
            "index_size_bytes": index_length,
            "table_size_bytes": table_size_bytes,
            "avg_row_size_bytes": avg_row_size_bytes,
        }
    return stats


def _estimate_restricted_size_bytes(row_count: int, storage_stats: dict):
    if row_count == 0:
        return 0

    avg_row_size_bytes = storage_stats.get("avg_row_size_bytes")
    if avg_row_size_bytes is None:
        return None

    return int(round(avg_row_size_bytes * row_count))


def _find_root_tables(all_tables):
    table_set = set(all_tables)
    roots = []

    for table_name in all_tables:
        tbl = dj.FreeTable(dj.conn(), table_name)
        try:
            internal_parents = [
                p for p in tbl.parents(as_objects=False)
                if p in table_set
            ]
        except Exception:
            # If dependency metadata fails, treat as root and continue.
            internal_parents = []

        if len(internal_parents) == 0:
            roots.append(table_name)

    return sorted(roots)


def _restrict_root_table(
    root_table_name,
    dataset_restriction,
    label_restriction,
    set_restriction,
    use_set_restriction,
):
    root = dj.FreeTable(dj.conn(), root_table_name)
    columns = set(root.heading.names)
    rel = root
    modes = []

    # Apply direct restrictions when these fields are present in the root table.
    if "session_label" in columns:
        rel = rel & label_restriction
        modes.append("session_label")

    if use_set_restriction and "set_name" in columns:
        rel = rel & set_restriction
        modes.append("set_name")

    # Otherwise, fall back to dataset lineage restriction when possible.
    if not modes and "dataset" in columns:
        rel = rel & dataset_restriction
        modes.append("dataset")

    if not modes:
        modes.append("unrestricted")

    return rel, "+".join(modes)


def _apply_table_dataset_override(
    rel,
    table_name: str,
    prefix: str,
    table_dataset_overrides: dict,
):
    table_key = _table_suffix_key(table_name, prefix)
    override_dataset = table_dataset_overrides.get(table_key)
    if override_dataset is None:
        return rel, None

    if "dataset" not in rel.heading.names:
        return rel, None

    return rel & [{"dataset": override_dataset}], override_dataset


def _safe_descendants(root_name, all_table_set):
    try:
        descendants = dj.FreeTable(dj.conn(), root_name).descendants(as_objects=False)
        descendants = [d for d in descendants if d in all_table_set]
        descendants = sorted(set([root_name] + descendants))
    except Exception:
        # If root is absent in dependency graph, process only itself.
        descendants = [root_name]
    return descendants


def collect_tables_for_target_labels(
    session_labels,
    set_labels=None,
    schema_suffixes=SCHEMA_SUFFIXES,
    included_tables=None,
    table_dataset_overrides=None,
    fetch_keys=False,
    max_keys_per_table=None,
    skip_zero=False,
):
    set_labels = [] if set_labels is None else set_labels
    included_tables = INCLUDED_TABLES_FROM_FILE if included_tables is None else included_tables
    table_dataset_overrides = TABLE_DATASET_OVERRIDES if table_dataset_overrides is None else table_dataset_overrides

    label_restriction = [{"session_label": label} for label in session_labels]
    set_restriction = [{"set_name": s} for s in set_labels]
    use_set_restriction = len(set_restriction) > 0

    schema_databases, prefix = _infer_schema_databases(
        base_dataset_database=VRDataset.database,
        suffixes=schema_suffixes,
    )
    effective_included_tables = _apply_database_prefix_to_included_tables(prefix, included_tables)
    all_tables = _list_schema_tables(schema_databases, included_tables=effective_included_tables)
    all_table_set = set(all_tables)
    root_tables = _find_root_tables(all_tables)
    table_storage_stats = _fetch_table_storage_stats(schema_databases)

    dataset_table_name = _full_table_name(f"{prefix}vr4mice", "dataset")
    dataset_restriction = dj.FreeTable(dj.conn(), dataset_table_name) & label_restriction

    # Apply set_name restriction through decision.session_label lookup
    # so we do not depend on decision.experiment_member being populated.
    session_label_table_name = _full_table_name(f"{prefix}decision", "session_label")
    if use_set_restriction and session_label_table_name in all_table_set:
        session_label_rel = dj.FreeTable(dj.conn(), session_label_table_name)
        session_label_rel = session_label_rel & set_restriction
        dataset_restriction = dataset_restriction & session_label_rel.proj("session_label")

    best_records = {}
    table_relations = {}

    for root_name in root_tables:
        root_rel, mode = _restrict_root_table(
            root_name,
            dataset_restriction=dataset_restriction,
            label_restriction=label_restriction,
            set_restriction=set_restriction,
            use_set_restriction=use_set_restriction,
        )

        root_count = len(root_rel)
        if skip_zero and root_count == 0:
            continue

        # Restrict descendants using root primary key attributes only to avoid
        # dependent-attribute collisions (e.g., shared secondary field 'description').
        root_key_rel = root_rel.proj(*root_rel.primary_key)

        descendants = _safe_descendants(root_name, all_table_set)

        for table_name in descendants:
            tbl = dj.FreeTable(dj.conn(), table_name)
            rel = tbl & root_key_rel
            rel, override_dataset = _apply_table_dataset_override(
                rel,
                table_name=table_name,
                prefix=prefix,
                table_dataset_overrides=table_dataset_overrides,
            )
            count = len(rel)

            if skip_zero and count == 0:
                continue

            storage_stats = table_storage_stats.get(table_name, {})
            restricted_size_estimate_bytes = _estimate_restricted_size_bytes(count, storage_stats)
            table_size_bytes = storage_stats.get("table_size_bytes")
            size_bytes = restricted_size_estimate_bytes if restricted_size_estimate_bytes is not None else table_size_bytes
            record_mode = mode if override_dataset is None else f"{mode}+dataset_override"

            record = {
                "schema": _schema_from_full_table_name(table_name),
                "root_table": root_name,
                "table_name": table_name,
                "restriction_mode": record_mode,
                "dataset_override": override_dataset,
                "count": count,
                "primary_key": tbl.primary_key,
                "size_go": round(size_bytes / (1024 ** 3), 3) if size_bytes is not None else None,
            }

            if fetch_keys:
                if max_keys_per_table is None:
                    record["keys"] = rel.fetch("KEY")
                else:
                    record["keys"] = rel.fetch("KEY", limit=max_keys_per_table)

            # Keep only one record per table: prefer explicit table overrides, then
            # restricted rows, then larger counts.
            current = best_records.get(table_name)
            current_priority = current.get("_priority") if current is not None else None
            override_requested = _table_suffix_key(table_name, prefix) in table_dataset_overrides
            new_priority = (
                override_requested and override_dataset is None,
                mode == "unrestricted",
                -count,
            )

            if current is None or new_priority < current_priority:
                record["_priority"] = new_priority
                best_records[table_name] = record
                table_relations[table_name] = rel

    summary_df = pd.DataFrame(best_records.values())
    if not summary_df.empty:
        summary_df = summary_df.drop(columns=["_priority"], errors="ignore")
        summary_df = summary_df.sort_values(
            ["schema", "table_name"]
        ).reset_index(drop=True)

    return {
        "summary": summary_df,
        "relations": table_relations,
        "root_tables": root_tables,
        "all_tables": all_tables,
        "effective_included_tables": effective_included_tables,
        "schema_databases": schema_databases,
    }


# 3) Run
res = collect_tables_for_target_labels(
    session_labels=SESSION_LABELS,
    set_labels=SET_LABELS,
    fetch_keys=False,      # True if you want to materialize KEY dicts
    max_keys_per_table=100 # only used when fetch_keys=True
)

print(f"Schemas scanned: {res['schema_databases']}")
print(f"Total schema tables scanned: {len(res['all_tables'])}")
print(f"Include full-table list loaded from file: {len(INCLUDED_TABLES_FROM_FILE)}")
print(f"Effective include full-table list size: {len(res['effective_included_tables'])}")
print(f"Table-specific dataset overrides: {TABLE_DATASET_OVERRIDES}")
print(f"Root tables discovered: {len(res['root_tables'])}")
print(f"Rows in summary (including zero-count): {len(res['summary'])}")
print(f"Duplicate table_name rows in summary: {res['summary'].duplicated(['table_name']).sum()}")

# Quick check by schema
if not res["summary"].empty:
    display(res["summary"].groupby("schema", as_index=False)["count"].sum())

Schemas scanned: ['vr4mice', 'base', 'base_analysis', 'decision', 'dlc', 'interpolated_trajectories', 'latency_tests', 'session_metrics']
Total schema tables scanned: 24
Include full-table list loaded from file: 24
Effective include full-table list size: 24
Table-specific dataset overrides: {'dlc.__offline_kinematics': 'Pheasant_2024-08-21_1', 'latency_tests.__signals_photodiode_aligned': 'Latencytest1_2024-10-31_2'}
Root tables discovered: 11
Rows in summary (including zero-count): 24
Duplicate table_name rows in summary: 0


,schema,count
0,base_analysis,906
1,decision,1556
2,dlc,1
3,interpolated_trajectories,1780
4,latency_tests,84
5,session_metrics,906
6,vr4mice,1172


In [23]:
pd.set_option('display.max_rows', None)

In [24]:
res["summary"]

,schema,root_table,table_name,restriction_mode,dataset_override,count,primary_key,size_go
0,base_analysis,`vr4mice`.`dataset`,`base_analysis`.`__box_data_frame`,session_label,None,453,[dataset],0.000
1,base_analysis,`vr4mice`.`dataset`,`base_analysis`.`__data_frame`,session_label,None,453,[dataset],42.845
2,decision,`decision`.`#label_set`,`decision`.`#label_set`,unrestricted,None,8,[label_set_id],0.000
3,decision,`decision`.`#label`,`decision`.`#label`,unrestricted,None,14,[label_key],0.000
4,decision,`decision`.`__decision_points`,`decision`.`__decision_points`,set_name,None,10,"[label_set_id, param_id, set_name, stage_name,...",0.000
5,decision,`decision`.`__inclusion_status`,`decision`.`__inclusion_status`,dataset,None,471,[dataset],0.000
6,decision,`vr4mice`.`dataset`,`decision`.`__prediction_model__session_predic...,session_label,None,558,"[label_set_id, param_id, set_name, stage_name,...",0.000
7,decision,`decision`.`__prediction_model`,`decision`.`__prediction_model`,set_name,None,24,"[label_set_id, param_id, set_name, stage_name]",0.000
8,decision,`vr4mice`.`dataset`,`decision`.`_experiment_member`,session_label,None,471,[dataset],0.000
9,dlc,`dlc`.`__offline_kinematics`,`dlc`.`__offline_kinematics`,dataset+dataset_override,Pheasant_2024-08-21_1,1,"[dataset, camera, doe, model_name]",0.007


In [ ]:
res["summary"]["size_go"].sum()

44.509